# Transformer fairness evaluation: including misgendering

Computes baseline subgroup accuracy, FPR, TPR, and gap metrics for the transformer models.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================
# Unified evaluation script for Table 8 models
# - Longformer (GEP base)
# - MSTL Longformer (SST + GEP)
# - BERT
# - ClinicalBERT
# - Handles 4096-token docs for all
# - Computes overall & subgroup metrics + ΔFPR / ΔTPR
# ============================================

import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---------------------------
# Saved model paths
# ---------------------------
TEST_CSV = str(DATA_DIR / 'GEP_test_80_20.csv')

LONGFORMER_BASE_DIR = str(MODEL_DIR / 'Longformer_GEP_Base_80_20')
LONGFORMER_MSTL_DIR = str(MODEL_DIR / 'Longformer_berkeley_phenotype_mimic_GEP_80_20')
BERT_DIR            = str(MODEL_DIR / 'BERT_GEP')
CLINICALBERT_DIR    = str(MODEL_DIR / 'clinicalbert_GEP')

# ---------------------------
# Load test data
# ---------------------------
test_df = pd.read_csv(TEST_CSV)
texts  = test_df["text"].astype(str).tolist()
labels = test_df["label"].astype(int).to_numpy()
geps   = test_df["GEP"].astype(int).to_numpy()  # 0 = NGEP, 1 = GEP

# ---------------------------
# BERT-style chunked dataset (4096 via chunks)
# ---------------------------
class ChunkedTextDataset(Dataset):
    """
    For BERT / ClinicalBERT:
    - Tokenize without truncation
    - Cap at doc_max_length (e.g., 4096)
    - Split into 510-token chunks and wrap with [CLS]/[SEP]
    - Pad each chunk to 512
    """
    def __init__(self, texts, labels, tokenizer,
                 chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        token_ids = self.tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False
        )
        token_ids = token_ids[: self.doc_max_length]

        chunks = []
        for i in range(0, len(token_ids), self.chunk_size):
            core = token_ids[i:i+self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + core + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk = chunk + [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            else:
                chunk = chunk[: self.max_length]
            chunks.append(chunk)

        if len(chunks) == 0:
            # empty doc fallback
            chunk = [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id]
            chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks = [chunk]
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        chunks = self.chunk_text(text)
        return {
            "chunks": torch.tensor(chunks, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.float),
            "num_chunks": len(chunks)
        }

def bert_collate_fn(batch):
    all_chunks = [item["chunks"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.float)
    num_chunks = [item["num_chunks"] for item in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {
        "chunks": flat_chunks,
        "labels": labels,
        "num_chunks": num_chunks
    }

# ---------------------------
# Longformer full-sequence dataset (4096 tokens)
# ---------------------------
class LongformerTextDataset(Dataset):
    """
    For Longformer-based models:
    - Tokenize with truncation at max_length (4096)
    - No chunking
    """
    def __init__(self, texts, labels, tokenizer, max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )

        input_ids = torch.tensor(enc["input_ids"], dtype=torch.long)
        attention_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.float)
        }

def longformer_collate_fn(tokenizer):
    pad_id = tokenizer.pad_token_id

    def _collate(batch):
        input_ids_list = [b["input_ids"] for b in batch]
        attn_list      = [b["attention_mask"] for b in batch]
        labels         = torch.stack([b["label"] for b in batch])

        padded_input_ids = pad_sequence(
            input_ids_list, batch_first=True, padding_value=pad_id
        )
        padded_attn_mask = pad_sequence(
            attn_list, batch_first=True, padding_value=0
        )

        # CLS (position 0) gets global attention
        global_attention_mask = torch.zeros_like(padded_input_ids)
        global_attention_mask[:, 0] = 1

        return {
            "input_ids": padded_input_ids,
            "attention_mask": padded_attn_mask,
            "global_attention_mask": global_attention_mask,
            "labels": labels
        }
    return _collate

# ---------------------------
# Evaluation helpers
# ---------------------------
def eval_chunked_model(model, tokenizer, texts, labels, batch_size=4):
    ds = ChunkedTextDataset(texts, labels, tokenizer,
                            chunk_size=510, max_length=512, doc_max_length=4096)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        collate_fn=bert_collate_fn,
                        pin_memory=torch.cuda.is_available())
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating (chunked)", ncols=120):
            chunks = batch["chunks"].to(DEVICE)
            labs   = batch["labels"].to(DEVICE)
            num_chunks = batch["num_chunks"]

            outputs = model(
                input_ids=chunks,
                attention_mask=(chunks != tokenizer.pad_token_id)
            )
            chunk_logits = outputs.logits.squeeze(-1)  # [sum_chunks]

            pooled_logits = []
            idx = 0
            for nc in num_chunks:
                doc_logits = chunk_logits[idx:idx+nc]
                pooled = torch.max(doc_logits)  # max pooling over chunks
                pooled_logits.append(pooled)
                idx += nc
            pooled_logits = torch.stack(pooled_logits)
            probs = torch.sigmoid(pooled_logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labs.cpu().numpy().astype(int))

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    return preds, all_labels, all_probs

def eval_longformer_model(model, tokenizer, texts, labels, batch_size=2):
    ds = LongformerTextDataset(texts, labels, tokenizer, max_length=4096)
    collate = longformer_collate_fn(tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        collate_fn=collate,
                        pin_memory=torch.cuda.is_available())
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating (Longformer)", ncols=120):
            input_ids        = batch["input_ids"].to(DEVICE)
            attention_mask   = batch["attention_mask"].to(DEVICE)
            global_attention = batch["global_attention_mask"].to(DEVICE)
            labs             = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention
            )
            logits = outputs.logits.squeeze(-1)
            probs  = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labs.cpu().numpy().astype(int))

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    return preds, all_labels, all_probs

# ---------------------------
# Metric & fairness computation
# ---------------------------
def compute_metrics_and_fairness(name, y_true, y_pred, y_prob, y_gep):
    print(f"\n========== {name} ==========")

    # Overall metrics
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    print(f"Overall: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

    # Group-wise metrics
    for g, label in [(0, "GEP=0 (NGEP)"), (1, "GEP=1 (GEP)")]:
        mask = (y_gep == g)
        if mask.sum() == 0:
            print(f"{label}: N=0 (no samples)")
            continue
        acc_g = accuracy_score(y_true[mask], y_pred[mask])
        prec_g, rec_g, f1_g, _ = precision_recall_fscore_support(
            y_true[mask], y_pred[mask], average="binary", zero_division=0
        )
        try:
            auc_g = roc_auc_score(y_true[mask], y_prob[mask])
        except ValueError:
            auc_g = np.nan
        print(f"{label}: N={mask.sum()}, Acc={acc_g:.4f}, Prec={prec_g:.4f}, Rec={rec_g:.4f}, F1={f1_g:.4f}, AUC={auc_g:.4f}")

    # ΔFPR / ΔTPR
    def group_rates(y, yhat, g):
        res = {}
        for gg in [0, 1]:
            m = (g == gg)
            if m.sum() == 0:
                res[gg] = {"TPR": np.nan, "FPR": np.nan}
                continue
            tn, fp, fn, tp = confusion_matrix(y[m], yhat[m], labels=[0, 1]).ravel()
            TPR = tp / max(1, tp + fn)
            FPR = fp / max(1, fp + tn)
            res[gg] = {"TPR": TPR, "FPR": FPR}
        dFPR = abs(
            (0 if np.isnan(res[0]["FPR"]) else res[0]["FPR"]) -
            (0 if np.isnan(res[1]["FPR"]) else res[1]["FPR"])
        )
        dTPR = abs(
            (0 if np.isnan(res[0]["TPR"]) else res[0]["TPR"]) -
            (0 if np.isnan(res[1]["TPR"]) else res[1]["TPR"])
        )
        return res, dFPR, dTPR

    rates, dFPR, dTPR = group_rates(y_true, y_pred, y_gep)
    print(f"ΔFPR = {dFPR*100:.2f} pp, ΔTPR = {dTPR*100:.2f} pp")
    print("Group rates:")
    for g in [0, 1]:
        print(f"  GEP={g}: TPR={rates[g]['TPR']:.4f}, FPR={rates[g]['FPR']:.4f}")

    return {
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1,
        "auc": auc,
        "rates": rates,
        "delta_FPR": dFPR,
        "delta_TPR": dTPR,
    }

# ---------------------------
# 1) Longformer (GEP base)
# ---------------------------
print("\n=== Evaluating Longformer (GEP base) ===")
lf_tokenizer_base = AutoTokenizer.from_pretrained(LONGFORMER_BASE_DIR)
lf_model_base = AutoModelForSequenceClassification.from_pretrained(
    LONGFORMER_BASE_DIR
).to(DEVICE)

lf_base_preds, lf_base_labels, lf_base_probs = eval_longformer_model(
    lf_model_base, lf_tokenizer_base, texts, labels, batch_size=2
)
_ = compute_metrics_and_fairness(
    "Longformer (GEP base)", lf_base_labels, lf_base_preds, lf_base_probs, geps
)

# ---------------------------
# 2) MSTL Longformer (SST + GEP)
# ---------------------------
print("\n=== Evaluating MSTL Longformer (SST + GEP) ===")
lf_tokenizer_mstl = AutoTokenizer.from_pretrained(LONGFORMER_MSTL_DIR)
lf_model_mstl = AutoModelForSequenceClassification.from_pretrained(
    LONGFORMER_MSTL_DIR
).to(DEVICE)

lf_mstl_preds, lf_mstl_labels, lf_mstl_probs = eval_longformer_model(
    lf_model_mstl, lf_tokenizer_mstl, texts, labels, batch_size=2
)
_ = compute_metrics_and_fairness(
    "MSTL Longformer (SST+GEP)", lf_mstl_labels, lf_mstl_preds, lf_mstl_probs, geps
)

# ---------------------------
# 3) BERT (chunked 4096)
# ---------------------------
print("\n=== Evaluating BERT (GEP) ===")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_DIR)
if bert_tokenizer.pad_token_id is None:
    bert_tokenizer.pad_token = bert_tokenizer.sep_token

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_DIR
).to(DEVICE)

bert_preds, bert_labels, bert_probs = eval_chunked_model(
    bert_model, bert_tokenizer, texts, labels, batch_size=4
)
_ = compute_metrics_and_fairness(
    "BERT (GEP)", bert_labels, bert_preds, bert_probs, geps
)

# ---------------------------
# 4) ClinicalBERT (chunked 4096)
# ---------------------------
print("\n=== Evaluating ClinicalBERT (GEP) ===")
cb_tokenizer = AutoTokenizer.from_pretrained(CLINICALBERT_DIR)
if cb_tokenizer.pad_token_id is None:
    cb_tokenizer.pad_token = cb_tokenizer.sep_token

cb_model = AutoModelForSequenceClassification.from_pretrained(
    CLINICALBERT_DIR
).to(DEVICE)

cb_preds, cb_labels, cb_probs = eval_chunked_model(
    cb_model, cb_tokenizer, texts, labels, batch_size=4
)
_ = compute_metrics_and_fairness(
    "ClinicalBERT (GEP)", cb_labels, cb_preds, cb_probs, geps
)

print("\nAll evaluations complete.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams["font.family"] = "sans-serif"
# =========================
# Data
# =========================
models = ["Longformer\n(GEP base)", "MSTL\nLongformer", "BERT", "ClinicalBERT"]

# Accuracy (%)
acc_overall = np.array([0.7285, 0.8278, 0.7748, 0.7417]) * 100
acc_gep0    = np.array([0.7722, 0.8861, 0.8481, 0.7722]) * 100
acc_gep1    = np.array([0.6806, 0.7639, 0.6944, 0.7083]) * 100

# F1 (%)
f1_overall = np.array([0.7092, 0.8000, 0.7463, 0.7153]) * 100
f1_gep0    = np.array([0.6087, 0.7907, 0.6842, 0.6250]) * 100
f1_gep1    = np.array([0.7579, 0.8046, 0.7708, 0.7640]) * 100

# Fairness gaps (percentage points)
delta_fpr = np.array([31.51, 15.76, 43.38, 15.00])
delta_tpr = np.array([10.00,  7.22, 17.22,  0.56])

# =========================
# Color palette (your colors)
# =========================
palette = [
    "#ccebc5",  # Longformer (GEP base)
    "#7bccc4",  # MSTL Longformer
    "#2b8cbe",  # BERT
    "#084081",  # ClinicalBERT
]

gap_palette = [
    "#4eb3d3",  # ΔFPR
    "#0868ac",  # ΔTPR
]

# =========================
# Helper to make grouped bar plots
# =========================
def grouped_bar_plot(cluster_labels, values_per_cluster, ylabel, title):
    """
    cluster_labels: list of cluster names (e.g., ["Overall", "GEP=0", "GEP=1"])
    values_per_cluster: list of arrays, one per cluster, each len=models
    """
    n_clusters = len(cluster_labels)
    n_models = len(models)
    x = np.arange(n_clusters)
    width = 0.18  # bar width

    plt.figure(figsize=(9, 5))
    for i in range(n_models):
        offsets = x + (i - (n_models - 1) / 2) * width
        vals = [cluster[i] for cluster in values_per_cluster]
        plt.bar(offsets, vals, width, label=models[i], color=palette[i])

    plt.xticks(x, cluster_labels)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0, 100)
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# =========================
# Plot 1: Accuracy grouped by subgroup cluster
# =========================
acc_clusters = [acc_overall, acc_gep0, acc_gep1]
grouped_bar_plot(
    cluster_labels=["Overall", "GEP = 0 (NGEP)", "GEP = 1 (GEP)"],
    values_per_cluster=acc_clusters,
    ylabel="Accuracy (%)",
    title="Accuracy by Subgroup and Model",
)

# =========================
# Plot 2: F1 grouped by subgroup cluster
# =========================
f1_clusters = [f1_overall, f1_gep0, f1_gep1]
grouped_bar_plot(
    cluster_labels=["Overall", "GEP = 0 (NGEP)", "GEP = 1 (GEP)"],
    values_per_cluster=f1_clusters,
    ylabel="F1 Score (%)",
    title="F1 Score by Subgroup and Model",
)

# =========================
# Plot 3: Fairness gaps (ΔFPR, ΔTPR) grouped by metric cluster
# =========================
gap_clusters = [delta_fpr, delta_tpr]  # each array is len(models)
n_clusters = 2
n_models = len(models)
x = np.arange(n_clusters)
width = 0.18

plt.figure(figsize=(9, 5))
for i in range(n_models):
    offsets = x + (i - (n_models - 1) / 2) * width
    vals = [gap_clusters[c][i] for c in range(n_clusters)]
    plt.bar(offsets, vals, width, label=models[i], color=palette[i])

plt.xticks(x, ["ΔFPR", "ΔTPR"])
plt.ylabel("Gap (percentage points)")
plt.title("Group Fairness Gaps by Model (Lower is Better)")
plt.ylim(0, max(delta_fpr.max(), delta_tpr.max()) + 5)
plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Data (order: BERT, ClinicalBERT, Longformer base, MSTL Longformer)
# =========================
models = ["BERT", "ClinicalBERT", "Longformer\n(GEP base)", "MSTL\nLongformer"]

# ---- Baseline metrics from Table 8 ----
# Accuracy (%)
acc_overall_base = np.array([0.7748, 0.7417, 0.7285, 0.8278]) * 100
acc_gep0_base    = np.array([0.8481, 0.7722, 0.7722, 0.8861]) * 100
acc_gep1_base    = np.array([0.6944, 0.7083, 0.6806, 0.7639]) * 100

# F1 (%)
f1_overall_base = np.array([0.7463, 0.7153, 0.7092, 0.8000]) * 100
f1_gep0_base    = np.array([0.6842, 0.6250, 0.6087, 0.7907]) * 100
f1_gep1_base    = np.array([0.7708, 0.7640, 0.7579, 0.8046]) * 100

# Fairness gaps (percentage points)
delta_fpr_base = np.array([43.38, 15.00, 31.51, 15.76])
delta_tpr_base = np.array([17.22,  0.56, 10.00,  7.22])

# ---- Equalized-odds–constrained "best fair" metrics ----
# Accuracy (%)
acc_overall_fair = np.array([74.17, 71.52, 78.15, 83.44])
acc_gep0_fair    = np.array([72.15, 74.68, 75.95, 86.08])
acc_gep1_fair    = np.array([76.39, 68.06, 80.56, 80.56])

# Fairness gaps (percentage points)
delta_fpr_fair = np.array([4.83, 7.60, 11.61, 6.65])
delta_tpr_fair = np.array([0.56, 1.67, 11.11, 0.00])

# =========================
# Color palette (light → dark)
# BERT (lightest), ClinicalBERT, Longformer, MSTL (darkest)
# =========================
palette_models = [
    "#f7fcf0",  # BERT
    "#ccebc5",  # ClinicalBERT
    "#7bccc4",  # Longformer base
    "#084081",  # MSTL Longformer
]

# Same hue but slightly darker for "fair" versions
palette_fair = [
    "#e0f3db",  # BERT fair
    "#a8ddb5",  # ClinicalBERT fair
    "#4eb3d3",  # Longformer fair
    "#0868ac",  # MSTL fair
]

# =========================
# Helper: grouped bar plot for baseline metrics
# =========================
def grouped_bar_plot_baseline(cluster_labels, values_per_cluster, ylabel, title):
    """
    cluster_labels: e.g. ["Overall", "GEP=0", "GEP=1"]
    values_per_cluster: list of arrays, one per cluster (len == n_models)
    """
    n_clusters = len(cluster_labels)
    n_models = len(models)
    x = np.arange(n_clusters)
    width = 0.18

    plt.figure(figsize=(9, 5))
    for i in range(n_models):
        offsets = x + (i - (n_models - 1) / 2) * width
        vals = [cluster[i] for cluster in values_per_cluster]
        plt.bar(offsets, vals, width, label=models[i], color=palette_models[i])

    plt.xticks(x, cluster_labels)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0, 100)
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# =========================
# Baseline plots: Accuracy & F1
# =========================
acc_clusters_base = [acc_overall_base, acc_gep0_base, acc_gep1_base]
grouped_bar_plot_baseline(
    cluster_labels=["Overall", "GEP = 0 (NGEP)", "GEP = 1 (GEP)"],
    values_per_cluster=acc_clusters_base,
    ylabel="Accuracy (%)",
    title="Baseline Accuracy by Subgroup and Model",
)

f1_clusters_base = [f1_overall_base, f1_gep0_base, f1_gep1_base]
grouped_bar_plot_baseline(
    cluster_labels=["Overall", "GEP = 0 (NGEP)", "GEP = 1 (GEP)"],
    values_per_cluster=f1_clusters_base,
    ylabel="F1 Score (%)",
    title="Baseline F1 Score by Subgroup and Model",
)

# =========================
# Fairness comparison: Accuracy (baseline vs best fair)
# =========================
def grouped_bar_plot_fair(cluster_labels, base_values, fair_values, ylabel, title):
    """
    For each subgroup cluster, show baseline vs fair bars per model.
    base_values / fair_values: [overall, gep0, gep1] arrays (len == n_models)
    """
    n_clusters = len(cluster_labels)
    n_models = len(models)
    x = np.arange(n_clusters)
    width = 0.08  # narrow bars to fit baseline+fair

    plt.figure(figsize=(10, 5))
    for i in range(n_models):
        offsets_base = x + (i - (n_models - 1) / 2) * 2 * width
        offsets_fair = offsets_base + width

        base_vals = [base_values[c][i] for c in range(n_clusters)]
        fair_vals = [fair_values[c][i] for c in range(n_clusters)]

        plt.bar(offsets_base, base_vals, width,
                color=palette_models[i], alpha=0.9)
        plt.bar(offsets_fair, fair_vals, width,
                color=palette_fair[i], alpha=0.9)

    plt.xticks(x, cluster_labels)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0, 100)

    # Legend: one pair for baseline vs fair
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=palette_models[0]),
        plt.Rectangle((0, 0), 1, 1, color=palette_fair[0]),
    ]
    labels = ["Baseline threshold", "Equalized-odds fair threshold"]
    plt.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

acc_clusters_fair_base = [acc_overall_base, acc_gep0_base, acc_gep1_base]
acc_clusters_fair_new  = [acc_overall_fair, acc_gep0_fair, acc_gep1_fair]

grouped_bar_plot_fair(
    cluster_labels=["Overall", "GEP = 0 (NGEP)", "GEP = 1 (GEP)"],
    base_values=acc_clusters_fair_base,
    fair_values=acc_clusters_fair_new,
    ylabel="Accuracy (%)",
    title="Accuracy Trade-offs: Baseline vs Equalized-Odds–Constrained Thresholds",
)

# =========================
# Fairness comparison: ΔFPR & ΔTPR (baseline vs best fair)
# =========================
x = np.arange(len(models))
width = 0.2

plt.figure(figsize=(9, 5))
# ΔFPR
plt.bar(x - width*1.5, delta_fpr_base, width,
        label="ΔFPR (baseline)", color="#f7fcf0")
plt.bar(x - width*0.5, delta_fpr_fair, width,
        label="ΔFPR (fair)", color="#e0f3db")
# ΔTPR
plt.bar(x + width*0.5, delta_tpr_base, width,
        label="ΔTPR (baseline)", color="#ccebc5")
plt.bar(x + width*1.5, delta_tpr_fair, width,
        label="ΔTPR (fair)", color="#7bccc4")

plt.xticks(x, models)
plt.ylabel("Gap (percentage points)")
plt.title("Group Fairness Gaps Before and After Threshold Optimization")
plt.ylim(0, max(delta_fpr_base.max(), delta_fpr_fair.max()) + 5)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Export paired predictions from the same frozen fitted models.
# Training predictions are used for threshold selection; testing predictions are
# used only after the selected thresholds have been frozen.
training_df = pd.read_csv(DATA_DIR / "GEP_train_80_20.csv")
training_texts = training_df["text"].astype(str).tolist()
training_labels = training_df["label"].astype(int).to_numpy()
training_geps = training_df["GEP"].astype(int).to_numpy()

prediction_dir = RESULTS_DIR / "predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)

def save_prediction_pair(model_slug, train_probabilities, test_probabilities):
    pd.DataFrame({
        "label": training_labels,
        "GEP": training_geps,
        "probability": train_probabilities,
    }).to_csv(
        prediction_dir / f"{model_slug}_including_misgendering_training.csv", index=False
    )
    pd.DataFrame({
        "label": labels,
        "GEP": geps,
        "probability": test_probabilities,
    }).to_csv(
        prediction_dir / f"{model_slug}_including_misgendering_testing.csv", index=False
    )

_, _, lf_base_train_probs = eval_longformer_model(
    lf_model_base, lf_tokenizer_base, training_texts, training_labels, batch_size=2
)
save_prediction_pair("longformer", lf_base_train_probs, lf_base_probs)

_, _, lf_mstl_train_probs = eval_longformer_model(
    lf_model_mstl, lf_tokenizer_mstl, training_texts, training_labels, batch_size=2
)
save_prediction_pair("mstl", lf_mstl_train_probs, lf_mstl_probs)

_, _, bert_train_probs = eval_chunked_model(
    bert_model, bert_tokenizer, training_texts, training_labels, batch_size=4
)
save_prediction_pair("bert", bert_train_probs, bert_probs)

_, _, clinicalbert_train_probs = eval_chunked_model(
    cb_model, cb_tokenizer, training_texts, training_labels, batch_size=4
)
save_prediction_pair("clinicalbert", clinicalbert_train_probs, cb_probs)

print(f"Saved paired training/testing probability files to {prediction_dir}")
